# DINOv3-S control 학습

기존 64crop DINO best.pt(e36, SHA 1bec54df97dd…)에서 추가 학습합니다. 백본 고정, patch 480, crop 64, TTA4.


## 설정


In [ ]:
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT=Path('/content/drive/MyDrive/KoMaP2026')
DATA_ROOT=Path('/content/komap_advanced_ram_data')
SOURCE_CHECKPOINT=PROJECT_ROOT/'runs'/'advanced_dinov3s16_ram_64patch_01'/'best.pt'
EXPECTED_SOURCE_SHA_PREFIX='1bec54df97dd'
EXPECTED_SOURCE_EPOCH=36
RUN_NAME='dinov3_hard_ablation_control_01'
PATCH_SIZE=480
BATCH_SIZE=8
PATCHES_PER_IMAGE=64
EPOCHS=20
DECODER_LR=1e-4
HARD_CROP_FRACTION=0.0
OHEM_FRACTION=1.0
NUM_WORKERS=2
EVAL_EVERY=2
EARLY_STOPPING_PATIENCE=0
MIN_EPOCHS=20
STOP_AFTER_EPOCH=None
RESUME=True
ABSENT_POLICY='ignore'


## 환경·코드


In [ ]:
import os, sys, importlib
from pathlib import Path
_candidates = [Path(os.environ['KOMAP_SUBMISSION_ROOT'])] if os.environ.get('KOMAP_SUBMISSION_ROOT') else []
_candidates += [Path.cwd(), Path('/content/KoMaP2026_Submission_DirectInference_20261015'),
                Path('/content/drive/MyDrive/KoMaP2026_Submission_DirectInference_20261015')]
CODE_ROOT = next((p.resolve() for p in _candidates if (p/'komap_submission_environment.py').is_file() and (p/'komap_submission_infer.py').is_file() and 'CURRENT_EXPORT_SHA256' in (p/'komap_submission_infer.py').read_text(encoding='utf-8')), None)
if CODE_ROOT is None:
    raise FileNotFoundError('제출 코드 폴더를 찾지 못했습니다. 00_제출파일_실행안내.ipynb에서 KOMAP_SUBMISSION_ROOT 경로를 설정하세요.')
import subprocess
subprocess.check_call([sys.executable, str(CODE_ROOT/'komap_submission_environment.py'), '--install'])


## 환경·코드


In [ ]:
import os, sys, importlib
from pathlib import Path
_candidates = [Path(os.environ['KOMAP_SUBMISSION_ROOT'])] if os.environ.get('KOMAP_SUBMISSION_ROOT') else []
_candidates += [Path.cwd(), Path('/content/KoMaP2026_Submission_DirectInference_20261015'),
                Path('/content/drive/MyDrive/KoMaP2026_Submission_DirectInference_20261015')]
CODE_ROOT = next((p.resolve() for p in _candidates if (p/'komap_submission_environment.py').is_file() and (p/'komap_submission_infer.py').is_file() and 'CURRENT_EXPORT_SHA256' in (p/'komap_submission_infer.py').read_text(encoding='utf-8')), None)
if CODE_ROOT is None:
    raise FileNotFoundError('제출 코드 폴더를 찾지 못했습니다. 00_제출파일_실행안내.ipynb에서 KOMAP_SUBMISSION_ROOT 경로를 설정하세요.')
CODE_DIR = CODE_ROOT / '.'
sys.dont_write_bytecode = True

for _module_name in list(sys.modules):
    if _module_name.startswith('komap_'):
        del sys.modules[_module_name]
sys.path[:] = [p for p in sys.path if p not in (str(CODE_ROOT), str(CODE_ROOT/'legacy_yolo_initial'))]
sys.path.insert(0, str(CODE_DIR))
importlib.invalidate_caches()
print('제출 코드:', CODE_DIR)


## 데이터·초기 모델


In [ ]:
import komap_specialist_data as kd
import komap_phase_fusion as pf
import komap_ablation_ram as kr
for module in (kd,pf,kr):
    importlib.reload(module)
    if Path(module.__file__).resolve().parent!=CODE_DIR.resolve():
        raise RuntimeError('다른 코드가 로드되었습니다. 새 런타임에서 실행하세요.')
RAM,DATA_REPORT=kr.prepare_ram(PROJECT_ROOT,DATA_ROOT)
SOURCE_INFO=kr.prepare_source_checkpoint(SOURCE_CHECKPOINT,DATA_REPORT,
    EXPECTED_SOURCE_SHA_PREFIX,EXPECTED_SOURCE_EPOCH)


## 학습·검증


In [ ]:
CONFIG=kr.Config(architecture='dinov3',
    pretrained_model_id=SOURCE_INFO['source_config']['pretrained_model_id'],
    patch=PATCH_SIZE,batch=BATCH_SIZE,patches_per_image=PATCHES_PER_IMAGE,
    epochs=EPOCHS,encoder_lr=1e-5,decoder_lr=DECODER_LR,
    freeze_encoder_epochs=0,warmup_epochs=1,eval_every=EVAL_EVERY,
    eval_stride=PATCH_SIZE//2,hard_crop_fraction=HARD_CROP_FRACTION,
    ohem_fraction=OHEM_FRACTION,validation_tta_rotations=4,
    absent_policy=ABSENT_POLICY)
RUN_DIR=PROJECT_ROOT/'runs'/RUN_NAME
HARD_COORDINATES=kr.prepare_hard_coordinates(RAM,DATA_REPORT,SOURCE_INFO,CONFIG,RUN_DIR)
BEST_PATH=kr.train(RAM,DATA_REPORT,RUN_DIR,CONFIG,num_workers=NUM_WORKERS,
    stop_after_epoch=STOP_AFTER_EPOCH,patience=EARLY_STOPPING_PATIENCE,
    min_epochs=MIN_EPOCHS,log_every=100,resume=RESUME,
    initial_checkpoint=SOURCE_INFO['path'],hard_coordinates=HARD_COORDINATES)


## 확률 내보내기


In [ ]:
EXPORT_DIR=kr.export_best(RAM,DATA_REPORT,RUN_DIR,
    tta_rotations=4,compare_plain=False)
print('실험 best:',BEST_PATH)
print('확률·마스크:',EXPORT_DIR)
